# Part 3 — Predict no-shows
Test set is used for reporting only, not model/threshold selection. Run after Part 1.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score,
                             recall_score, f1_score, roc_auc_score)
from pathlib import Path
ROOT=Path.cwd()
df=pd.read_csv(ROOT/'appointments_clean.csv')


## 3.1 Features, split and majority-class baseline

In [ ]:
features=['age','distance_km','fee_inr','lead_days','prior_no_shows','reminder_sent',
          'booking_channel','visit_type']
X=df[features].copy()
X['reminder_sent']=X['reminder_sent'].map({'Yes':1,'No':0})
X=pd.get_dummies(X,columns=['booking_channel','visit_type'])
assert X.shape[1]==11
assert not X.isna().any().any()
y=df['no_show']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
print('Train:',len(X_train),'test:',len(X_test),'test no-shows:',int(y_test.sum()))
print('Always-attended baseline accuracy:',(y_test==0).mean())


## 3.2 Logistic regression, metrics and reminder odds

In [ ]:
scaler=StandardScaler()
X_train_s=scaler.fit_transform(X_train)
X_test_s=scaler.transform(X_test)
lr=LogisticRegression(max_iter=1000)
lr.fit(X_train_s,y_train)
def report(model,X_eval,y_eval,threshold=0.5):
    probs=model.predict_proba(X_eval)[:,1]
    pred=(probs>=threshold).astype(int)
    return {'accuracy':accuracy_score(y_eval,pred),
            'precision':precision_score(y_eval,pred,zero_division=0),
            'recall':recall_score(y_eval,pred,zero_division=0),
            'f1':f1_score(y_eval,pred,zero_division=0),
            'roc_auc':roc_auc_score(y_eval,probs),
            'flagged':int(pred.sum()),'confusion_matrix':confusion_matrix(y_eval,pred).tolist()}
print('Logistic test:',report(lr,X_test_s,y_test))
print('Coefficients:')
for name,coef in zip(X.columns,lr.coef_[0]): print(name,round(float(coef),4))
idx=X.columns.get_loc('reminder_sent')
coef=lr.coef_[0][idx]
print('OR per 1 SD:',np.exp(coef),'OR No -> Yes:',np.exp(coef/scaler.scale_[idx]),
      'reminder SD:',scaler.scale_[idx])
print('Associations only, not causal effects.')


## 3.3 Decision trees and overfitting

In [ ]:
tree_full=DecisionTreeClassifier(random_state=42)
tree_3=DecisionTreeClassifier(max_depth=3,random_state=42)
for name,tree in [('Full',tree_full),('Depth 3',tree_3)]:
    tree.fit(X_train,y_train)
    print(name,'train accuracy:',tree.score(X_train,y_train),
          'test accuracy:',tree.score(X_test,y_test),'depth:',tree.get_depth())
print('A large training-test gap is evidence of overfitting.')


## 3.4 Random forest and descriptive test comparison

In [ ]:
rf=RandomForestClassifier(n_estimators=100,random_state=42)
rf.fit(X_train,y_train)
print('Forest test:',report(rf,X_test,y_test))
print('Top three features:',sorted(zip(X.columns,rf.feature_importances_),
                                  key=lambda item:item[1],reverse=True)[:3])
comparison=pd.DataFrame([{'model':'Logistic regression',**report(lr,X_test_s,y_test)},
                         {'model':'Depth-3 tree',**report(tree_3,X_test,y_test)},
                         {'model':'Random forest',**report(rf,X_test,y_test)}])
print(comparison[['model','accuracy','recall','roc_auc']].to_string(index=False))
print('Comparison is descriptive; formal selection below uses training CV only.')


## 3.5 Cross-validated model and threshold selection, then one test evaluation

In [ ]:
grid=GridSearchCV(RandomForestClassifier(random_state=42),
                  {'n_estimators':[100,200],'max_depth':[3,5,8]},
                  cv=5,scoring='roc_auc',n_jobs=-1)
grid.fit(X_train,y_train)
print('Tuned forest:',grid.best_params_,'CV AUC:',grid.best_score_)
lr_pipe=Pipeline([('scaler',StandardScaler()),('model',LogisticRegression(max_iter=1000))])
candidates={'Logistic regression':lr_pipe,'Depth-3 tree':DecisionTreeClassifier(max_depth=3,random_state=42),
            'Random forest':RandomForestClassifier(n_estimators=100,random_state=42)}
cv_scores={name:cross_val_score(est,X_train,y_train,cv=5,scoring='roc_auc').mean()
           for name,est in candidates.items()}
cv_scores['Tuned forest']=grid.best_score_
print('Training CV AUC:',cv_scores)
winner_name=max(cv_scores,key=cv_scores.get)
winner=grid.best_estimator_ if winner_name=='Tuned forest' else candidates[winner_name]
print('Selected on training CV only:',winner_name)
oof=cross_val_predict(winner,X_train,y_train,cv=5,method='predict_proba')[:,1]
threshold_results=[]
for threshold in (0.5,0.35):
    pred=(oof>=threshold).astype(int)
    threshold_results.append({'threshold':threshold,
                              'precision':precision_score(y_train,pred,zero_division=0),
                              'recall':recall_score(y_train,pred,zero_division=0),
                              'f1':f1_score(y_train,pred,zero_division=0),
                              'flagged':int(pred.sum())})
print(pd.DataFrame(threshold_results).to_string(index=False))
selected=max(threshold_results,key=lambda item:item['f1'])['threshold']
print('Threshold selected by training OOF F1:',selected)
print('Recall matters: a missed warning leaves a doctor slot empty; a call takes staff time.')
winner.fit(X_train,y_train)
final_metrics=report(winner,X_test,y_test,selected)
print('FINAL test metrics (one evaluation):',final_metrics)
test_probs=winner.predict_proba(X_test)[:,1]
test_scores=pd.DataFrame({'appointment_id':df.loc[X_test.index,'appointment_id'].to_numpy(),
                          'no_show':y_test.to_numpy(),'risk_prob':test_probs,
                          'flag':(test_probs>=selected).astype(int)})
test_scores.to_csv(ROOT/'test_risk_scores.csv',index=False)
assert len(test_scores)==198
print('Saved 198 test scores. Front desk should set threshold based on calling capacity and cost of empty slots.')
